In [1]:
import os
import pandas as pd

FEATURE_DIR = "data/engineered_features"
SPLIT_DIR = "data/split_data"
os.makedirs(SPLIT_DIR, exist_ok=True)

asset_files = {
    "BTC": os.path.join(FEATURE_DIR, "BTC_features.csv"),
    "ETH": os.path.join(FEATURE_DIR, "ETH_features.csv"),
    "XRP": os.path.join(FEATURE_DIR, "XRP_features.csv"),
    "SOL": os.path.join(FEATURE_DIR, "SOL_features.csv"),
}

## Helper Function

In [2]:
def chronological_split(df: pd.DataFrame,
                        train_frac: float = 0.50,
                        val_frac: float = 0.25,
                        test_frac: float = 0.25):
    if abs(train_frac + val_frac + test_frac - 1.0) > 1e-9:
        raise ValueError("train_frac + val_frac + test_frac must sum to 1.")

    df = df.sort_values("datetime").reset_index(drop=True)

    n = len(df)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    train_df = df.iloc[:train_end].copy()
    val_df   = df.iloc[train_end:val_end].copy()
    test_df  = df.iloc[val_end:].copy()

    return train_df, val_df, test_df

## Split Each Asset and Save

In [3]:
split_summary = []
all_train, all_val, all_test = [], [], []

for asset, file_path in asset_files.items():
    df = pd.read_csv(file_path)
    df["datetime"] = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
    df = df.sort_values("datetime").reset_index(drop=True)

    train_df, val_df, test_df = chronological_split(df)

    # save per-asset files
    train_path = os.path.join(SPLIT_DIR, f"{asset}_train.csv")
    val_path   = os.path.join(SPLIT_DIR, f"{asset}_val.csv")
    test_path  = os.path.join(SPLIT_DIR, f"{asset}_test.csv")

    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)

    # store for pooled split sets
    all_train.append(train_df)
    all_val.append(val_df)
    all_test.append(test_df)

    split_summary.append({
        "asset": asset,
        "total_rows": len(df),
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "test_rows": len(test_df),
        "train_start": train_df["datetime"].min(),
        "train_end": train_df["datetime"].max(),
        "val_start": val_df["datetime"].min(),
        "val_end": val_df["datetime"].max(),
        "test_start": test_df["datetime"].min(),
        "test_end": test_df["datetime"].max(),
    })

    print(f"{asset} saved:")
    print(f"  Train: {len(train_df):,} rows -> {train_path}")
    print(f"  Val:   {len(val_df):,} rows -> {val_path}")
    print(f"  Test:  {len(test_df):,} rows -> {test_path}")
    print()

BTC saved:
  Train: 21,868 rows -> data/split_data/BTC_train.csv
  Val:   10,934 rows -> data/split_data/BTC_val.csv
  Test:  10,934 rows -> data/split_data/BTC_test.csv

ETH saved:
  Train: 21,868 rows -> data/split_data/ETH_train.csv
  Val:   10,934 rows -> data/split_data/ETH_val.csv
  Test:  10,934 rows -> data/split_data/ETH_test.csv

XRP saved:
  Train: 21,868 rows -> data/split_data/XRP_train.csv
  Val:   10,934 rows -> data/split_data/XRP_val.csv
  Test:  10,934 rows -> data/split_data/XRP_test.csv

SOL saved:
  Train: 21,868 rows -> data/split_data/SOL_train.csv
  Val:   10,934 rows -> data/split_data/SOL_val.csv
  Test:  10,934 rows -> data/split_data/SOL_test.csv



## Save Split Summary

In [4]:
split_summary_df = pd.DataFrame(split_summary)
split_summary_path = os.path.join(SPLIT_DIR, "split_summary.csv")
split_summary_df.to_csv(split_summary_path, index=False)

print(f"\nSplit summary saved to {split_summary_path}")
split_summary_df


Split summary saved to data/split_data/split_summary.csv


,asset,total_rows,train_rows,val_rows,test_rows,train_start,train_end,val_start,val_end,test_start,test_end
0,BTC,43736,21868,10934,10934,2021-01-04 00:00:00+00:00,2023-07-04 17:00:00+00:00,2023-07-04 18:00:00+00:00,2024-10-02 07:00:00+00:00,2024-10-02 08:00:00+00:00,2025-12-31 21:00:00+00:00
1,ETH,43736,21868,10934,10934,2021-01-04 00:00:00+00:00,2023-07-04 17:00:00+00:00,2023-07-04 18:00:00+00:00,2024-10-02 07:00:00+00:00,2024-10-02 08:00:00+00:00,2025-12-31 21:00:00+00:00
2,XRP,43736,21868,10934,10934,2021-01-04 00:00:00+00:00,2023-07-04 17:00:00+00:00,2023-07-04 18:00:00+00:00,2024-10-02 07:00:00+00:00,2024-10-02 08:00:00+00:00,2025-12-31 21:00:00+00:00
3,SOL,43736,21868,10934,10934,2021-01-04 00:00:00+00:00,2023-07-04 17:00:00+00:00,2023-07-04 18:00:00+00:00,2024-10-02 07:00:00+00:00,2024-10-02 08:00:00+00:00,2025-12-31 21:00:00+00:00
